# Load Libraries and Data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors

from shapely.geometry import Point
import statsmodels.formula.api as smf

from particles_path import *
from is_in_earth_shadow import *


#Load the cleaned data
clean_rad = pd.read_excel("clean_data_kp.xlsx")

#### Create geodataframe for geopandas

In [ ]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

geometry = [Point(xy) for xy in zip(clean_rad["lon"], clean_rad["lat"])]
geo_rad = gpd.GeoDataFrame(clean_rad, geometry=geometry, crs="EPSG:4326")

### Define a constant colormap to use for all visuals

Default: `managua` 

`managua` is very good for highlighting all points on the graph.  
Change to `Reds` if you would rather only easily see extreme high values. All low values become very hard to see on the light background

In [ ]:
COLOR = "managua"

# Compare Sensors to Sunlight

## Compare all X-ray sensors to sunlight

In [ ]:
shadows = [0, 1]
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps"]

fig, axes = plt.subplots(4, 2, figsize=(14, len(detectors) * 3.7), constrained_layout=True)

for i, detector in enumerate(detectors):
    vmin = geo_rad[detector].min()
    vmax = geo_rad[detector].max()
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = COLOR

    for j, shadow in enumerate(shadows):
        subset = geo_rad[geo_rad["in_shadow"] == shadow]

        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.75,
            column=detector,
            cmap=cmap,
            legend=False
        )
        axes[i, j].set_title(f"{detector} while `in_shadow` == {shadow}", fontsize="xx-large")
        axes[i, j].set_xlabel("Longtiude", fontsize="x-large")
        axes[i, j].set_ylabel("Latitude", fontsize="x-large")
    # Add one shared colorbar on the right side of the row
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[i, :], shrink=0.8, pad=0.02)
    cbar.set_label(detector, fontsize="x-large")

fig.suptitle(f"X-rays Per Second by detector and in_shadow",  fontsize=28)
plt.show()

### Analysis of X-ray to sunlight

It appears that, once again, all X-ray sensors follow the same trends. For all sensors, X-rays appear higher when `in_shadow == 0`. `xray0_ps` catches some higher values in shadow in the north, but those are all low enough energy that they aren't picked up by the remaining sensors. Throughout the night, there are still noticable spikes, but never anything lasting for more than a single observation. Our highest values are found by `xray0_ps` the furthest north while `in_shadow == 0`. Interestingly, our few observations around the North Pole while `in_shadow == 1` show very similar readings to the rest of the planet barring the darker line over Greenland from `xray0_ps`.

## Compare the remaining sensors to sunlight

In [ ]:
shadows = [0, 1]
detectors = ["proton0_ps", "electron0_ps", "ses_ps", "total_radiation_ps"]

fig, axes = plt.subplots(len(detectors), 2, figsize=(14, len(detectors)*3.5), constrained_layout=True)

for i, detector in enumerate(detectors):
    # Compute global min/max across both shadow values for this detector
    vmin = geo_rad[geo_rad["group"]!=198][detector].min()                               # The "!= 198" removes an outlier that throws off the entire scale for ses_ps
    vmax = geo_rad[geo_rad["group"]!=198][detector].max()                               # The "!= 198" removes an outlier that throws off the entire scale for ses_ps
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap = COLOR

    for j, shadow in enumerate(shadows):
        subset = geo_rad[(geo_rad["group"] != 198) & (geo_rad["in_shadow"] == shadow)]  # The "!= 198" removes an outlier that throws off the entire scale for ses_ps
        world.plot(ax=axes[i, j], color="lightgrey")
        subset.plot(
            ax=axes[i, j],
            markersize=1,
            alpha=0.75,
            column=detector,
            cmap=cmap,
            norm=norm,       # enforce shared scale
            legend=False,    # suppress individual legends
        )
        axes[i, j].set_title(f"{detector} while `in_shadow` == {shadow}")

    # Add one shared colorbar on the right side of the row
    sm = ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes[i, :], shrink=0.8, pad=0.02)
    cbar.set_label(detector)

fig.suptitle("Remaining sensors per second by in_shadow", size="xx-large")
# plt.tight_layout()
plt.show()


### Analysis of remaining sensors to sunlight

- While it appears that protons are higher during the day, they are not anywhere near as dampened by shadow as X-rays are. Protons appear to be higher in the southern hemisphere, but they are not solely found there like X-rays appear to be with the northern hemisphere
- Electrons remain fairly low everywhere. From this quick glance, I'm not sure how much of an effect sunlight has on electron counts. Electrons do appear to be higher more frequently in the southern hemisphere, but it is not anywhere near as stark of a contrast as can be observed with X-rays between the hemispheres. Electrons seem to often follow similar trends to protons
- While `ses_ps` doesn't matter much to our analysis, it is worth pointing out that it is following relatively similar trends to protons and electrons
- Our total radiation column clearl highlights the increase in X-rays in the north and the increase of protons and electrons in the south. It also shows the lack of high radiation from any column around the equator, regardless of sunlight. Total radiation appears lower without sunlight, but a large part of that is due to X-rays being so much lower in shadow. It is difficult to make any conclusions with `total_radiation_ps` since it is a sum of very different kinds of radiation, but it is a great way to look at all of the trends simultaneously.

That line over Greenland when `in_shadow == 1` appears in all graphs. It might be worth looking into further once we start analyzing kp index

# View all data points for `in_shadow == 0` and `in_shadow == 1`

In [ ]:
particles_path(clean_rad[clean_rad["in_shadow"] == 0])
particles_path(clean_rad[clean_rad["in_shadow"] == 1])

### It's worth noting that we don't have any data directly over Antarctica at night, and a large majority of our data over the Arctic Circle is during the day. We should keep this sample disparity in mind for future analysis
Less important, but it is also worth noting that all of our data when `in_shadow == 0` is going one way on the plot, while all the data when `in_shadow == 1` is going the other way. This shows how consisntly the satellite was moving with the orbit of the earth.

# Inspect each sensor split by hemisphere and `in_shadow`

In [ ]:
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps", "ses_ps", "total_radiation_ps"]

for detector in detectors:
    subset = {
        "north, day": clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 0)][detector],
        "north, night": clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 1)][detector],
        "south, day": clean_rad[(clean_rad["lat"] < 0)  & (clean_rad["in_shadow"] == 0)][detector],
        "south, night": clean_rad[(clean_rad["lat"] < 0)  & (clean_rad["in_shadow"] == 1)][detector],
    }

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.boxplot(
        subset.values(),
        tick_labels=subset.keys(),
        patch_artist=True,
        boxprops=dict(facecolor="steelblue", alpha=0.6),
        medianprops=dict(color="black", linewidth=2),
        flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    ax.set_title(f"{detector} by hemisphere and in_shadow", fontsize="xx-large")
    ax.set_xlabel(f"Hemisphere and Illumination")
    ax.set_ylabel(f"{detector} (log)")
    ax.set_yscale("log")
    plt.tight_layout()
    plt.show()

## Analysis of boxplots
As we've already seen, we observe more X-rays in the northern hemisphere than the souther. We see more protons and electrons in the southern hemisphere. `ses_ps` appears to follow a very similar trend to the protons and electrons in all hemisphere and `in_shadow` combinations. Our total radiation appears to have a higher median in the sunlight than in shadow

### How many data points are in each region at each time?

In [ ]:
print(f"Data points in the northern hemisphere during the day: \t\t{len(clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 0)])}")
print(f"Data points in the northern hemisphere during the night: \t{len(clean_rad[(clean_rad["lat"] >= 0) & (clean_rad["in_shadow"] == 1)])}")
print(f"Data points in the southern hemisphere during the day: \t\t{len(clean_rad[(clean_rad["lat"] < 0) & (clean_rad["in_shadow"] == 0)])}")
print(f"Data points in the southern hemisphere during the night: \t{len(clean_rad[(clean_rad["lat"] < 0) & (clean_rad["in_shadow"] == 1)])}")

# Test for a statistically significant effect with latitude and in_shadow

In [ ]:
detectors = ["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps", "ses_ps", "total_radiation_ps"]

clean_rad = pd.read_excel("clean_data_kp.xlsx")

for detector in detectors:
    model = smf.ols(f"{detector} ~ lat * in_shadow", data=clean_rad).fit()
    # data = clean_rad[["lat", "in_shadow", "group", detector]].dropna().reset_index(drop=True)
    # model = smf.mixedlm(f"{detector} ~ lat * in_shadow", data=data, groups=data["group"]).fit()
    print(model.summary().as_text())
    print("\n\n\n")

### Claude results analysing OLS results (for future reference):

Here's a summary across all detectors:

**Direction of latitude effect (out of shadow)**

Interestingly, the detectors split into two groups:
- **xray0_ps, xray1_ps, xray2_ps, xray3_ps, total_radiation_ps** — positive lat coefficient, counts increase with latitude
- **proton0_ps, electron0_ps, ses_ps** — negative lat coefficient, counts decrease with latitude

This is physically meaningful — protons and electrons behave differently from X-rays under the influence of Earth's magnetic field.

**Does shadow kill the latitude effect?**

Consistently across all detectors, the `lat:in_shadow` coefficient has the opposite sign to `lat`, meaning shadow partially or fully cancels the latitude relationship. For xray0_ps specifically the effective in-shadow slope is nearly zero (~2), but for protons and electrons the cancellation is less complete.

**Model fit (R²)**

| Detector | R² |
|---|---|
| xray0_ps | 0.259 |
| electron0_ps | 0.102 |
| proton0_ps | 0.099 |
| total_radiation_ps | 0.146 |
| ses_ps | 0.045 |
| xray1_ps | 0.013 |
| xray2_ps | 0.012 |
| xray3_ps | 0.008 |

xray0_ps is by far the best explained by this model. xray1–3 and ses_ps have very low R², suggesting latitude and shadow alone don't capture much of their variance.

**Significance**
Every single coefficient is significant at p < 0.05 across all detectors, with one exception: `in_shadow` for xray3_ps has p=0.156, meaning shadow's main effect isn't significant for that detector (though the interaction still is).

**Autocorrelation**
Durbin-Watson is low across the board (0.2–0.6 for most), with xray3_ps being the best at 1.244. The autocorrelation concern applies to all of them, so the subsampling/HAC approach you used earlier would be worth applying to the full set.


### Claude resuls on mixed effects model:
Here's a summary of the key findings across all detectors:

**Compared to OLS, the mixed effects model tells a more conservative story**

The coefficients shrank considerably for most detectors, and several effects that looked significant under OLS are no longer significant here. This is exactly what you'd expect — OLS was overconfident because it ignored the group structure.

**Latitude effect**
Latitude is now significant for: proton0_ps, electron0_ps, ses_ps, total_radiation_ps, and xray0_ps. Notably, **xray1_ps and xray2_ps now show no significant latitude effect** (p=0.837 and p=0.885) — those findings from OLS were likely false positives driven by autocorrelation.

**Shadow effect**
Shadow remains significant for most detectors but also lost significance in a couple of places:
- **ses_ps in_shadow**: p=0.812 — not significant
- **electron0_ps lat:in_shadow**: p=0.064 — borderline
- **total_radiation_ps lat:in_shadow**: p=0.326 — not significant

**Convergence warning on xray3_ps**
That model failed to converge, so its results are unreliable and shouldn't be reported. This is likely because the Group Var is estimated at ~0, meaning the random effect is essentially zero — the groups don't explain much variance in xray3_ps. You could fall back to OLS with HAC standard errors for that detector specifically.

**Overall interpretation**
The detectors split into two tiers:
- **Strong, robust effects** (survived mixed effects): xray0_ps, proton0_ps, electron0_ps, total_radiation_ps
- **Weak or spurious effects**: xray1_ps, xray2_ps, xray3_ps, ses_ps — latitude/shadow explain very little once group structure is accounted for




## Overall interpretation (not AI)
#### During the day:  
- The further north you go, the more the more x-rays (from xray0) you will find.  
- The further south you go, the more protons and electrons you will find.

#### At night:
- Latitude does not affect electron0
- Proton0_ps seems to go in the other direction
- OLS (Ordinary Least Squares) shows in_shadow turning off latitude's affect, while MEM shows negative always.

#### Both latitude and in_shadow have an effect on particle counts

In [ ]:
detector="xray0_ps"

for shadow in [0, 1]:
    print(f"in_shadow == {shadow}\n\n")

    model = smf.ols(f"{detector} ~ lat", data=clean_rad[clean_rad["in_shadow"] == shadow]).fit()
    print(model.summary().as_text())

    data = clean_rad[["lat", "in_shadow", "group", detector]].dropna().reset_index(drop=True)
    data = data[data["in_shadow"] == shadow]
    model = smf.mixedlm(f"{detector} ~ lat", data=data, groups=data["group"]).fit()
    print(model.summary().as_text())
    print("\n\n\n")

#### The above chunk of code is used as another way to compare latitude and daylight. As far as I can tell, it continues to reflect the same findings listed above

# Compare polar circles

In [ ]:
arctic = clean_rad[clean_rad["lat"] >= 66.5]
antarctic = clean_rad[clean_rad["lat"] <= -66.5]

## Check how many data points we have in each circle in and out of sunlight

In [ ]:
for circle in [(arctic, "Arctic"), (antarctic, "Antarctic")]:
    for shadow in [0, 1]:
        print(f"Data points in {circle[1]} Circle when in {"shadow" if shadow else "sunlight"}: {len(circle[0][circle[0]["in_shadow"] == shadow])}")

We only have two data points in the Antarctic Circle in shadow, so we will not be analyzing it to avoid each point representing 10 million kilometers.

For the rest of this section, I will be analyzing both sunlight and shadow in the Arctic Circle and sunlight in the Antarctic Circle

## Compare radiation values for the remaining three

In [ ]:
arctic_day = arctic[arctic["in_shadow"] == 0]
arctic_night = arctic[arctic["in_shadow"] == 1]
antarctic_day = antarctic[antarctic["in_shadow"] == 0]

data = [arctic_day, arctic_night, antarctic_day]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), constrained_layout=True)

colors = ["blue", "orange", "green", "red", "purple", "pink"]

for i, df in enumerate(data):
    grouped = df.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]].mean()

    for j, column in enumerate(grouped.columns):
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second", color=colors[j])

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"}")
    axes[i].set_ylabel("Particles Per Second (Log)")

fig.suptitle("Group averages for all sensors", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="medium")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 6), constrained_layout=True)

for i, df in enumerate(data):
    grouped = df.groupby("group")[["xray0_ps", "proton0_ps", "electron0_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"}", fontsize="x-large")
    axes[i].set_ylabel("Particles Per Second (Log)")


fig.suptitle("Group averages for all sensors", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="medium")
plt.show()

### Analysis of line graphs
I was not expecting `xray0_ps` to follow the trends of `proton0_ps` and `electron0_ps` in the Antarctic Circle in sunlight. That is fascinating and absolutely something we should look into further. `xray0_ps` follows the trends of the other two in the Arctic circle as well, but it deviates more often in those graphs. 
 
The large gap in data in the Antarctic Circle is very evident here.

It looks like `proton0_ps` and `electron0_ps` dip way lower than `xray0_ps` and almost rhythmically in the Arctic Circle in the latter half of the data collection period. That's around when the X-rays become more prevalent in the Northern hemisphere. 

The other X-ray sensors still follow the trends of `xray0_ps`, though the values are all very, very low in comparison. This is consistent with what we've already seen.
 
It's interesting how high all of the values are in January

## Plot the same data in a box and whisker plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes = axes.flatten()

for i, df in enumerate(data):

    subset = df[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

    bp = axes[i].boxplot(subset,
                         tick_labels=subset.keys(),
                         patch_artist=True,
                         boxprops=dict(facecolor="steelblue", alpha=0.6),
                         medianprops=dict(color="black", linewidth=2),
                         flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    for median_line in bp['medians']:
        median_val = median_line.get_ydata()[0]
        left_edge_x = median_line.get_xdata()[0]

        axes[i].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)
    
    axes[i].set_yscale("log")
    axes[i].set_ylim(10**-3, 10**5)
    axes[i].set_xlabel("Radiation Sensor")
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(df)})")

fig.delaxes(axes[3])
plt.suptitle("All sensors per second by pole and daylight", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of box and whisker plot

This could be a sample size difference, but X-rays defintely appear to be most prevalent in the sunlight and in the northern hempiphere. It already appeared that way in previous analysis, but it's visible here as well.

Interestingly, `proton0_ps` are not super far apart between the Arctic and Antarctic circles in the daylight. 

Electrons are clearly higher in the southern hemisphere. They are also steady in the Arctic circle regardless of daylight.

# How does the radiation differ when it's dark on the ground?

### Create a column similar to `in_shadow` that reports if it is in shadow at sea level

In [ ]:
clean_rad['alt0'] = 0

clean_rad["ground_shadow"] = clean_rad.apply(
    lambda row: is_in_earth_shadow(
        row["lat"],
        row["lon"],
        row["alt0"],
        row["timestamp"]
    ),
    axis=1
).replace({True: 1, False: 0})

clean_rad.drop("alt0", axis=1, inplace=True)

In [ ]:
for circle in [(arctic, "Arctic"), (antarctic, "Antarctic")]:
    for shadow in [0, 1]:
        print(f"Data points in {circle[1]} Circle when in {"shadow" if shadow else "sunlight"}: {len(circle[0][circle[0]["in_shadow"] == shadow])}")

### Check the sizes of all combinations of `ground_shadow`, `in_shadow`, and the poles

In [ ]:
print(f"Size of Arctic satellite day:\t\t {len(clean_rad[(clean_rad["in_shadow"] == 0) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic ground day:\t\t {len(clean_rad[(clean_rad["ground_shadow"] == 0) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 1) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Arctic ground shadow:\t\t {len(clean_rad[(clean_rad["ground_shadow"] == 1) & (clean_rad["lat"] >= 66.5)])}")
print(f"Size of Antarctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 0) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic ground shadow:\t {len(clean_rad[(clean_rad["ground_shadow"] == 0) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic satellite shadow:\t {len(clean_rad[(clean_rad["in_shadow"] == 1) & (clean_rad["lat"] <= -66.5)])}")
print(f"Size of Antarctic ground shadow:\t {len(clean_rad[(clean_rad["ground_shadow"] == 1) & (clean_rad["lat"] <= -66.5)])}")


### Redefine our test dataframes now that we have a new column

In [ ]:
arctic = clean_rad[clean_rad["lat"] >= 66.5]
antarctic = clean_rad[clean_rad["lat"] <= -66.5]

ground_arctic_day = arctic[arctic["ground_shadow"] == 0]
ground_arctic_night = arctic[arctic["ground_shadow"] == 1]
ground_antarctic_day = antarctic[antarctic["ground_shadow"] == 0]
ground_antarctic_night = antarctic[antarctic["ground_shadow"] == 1]

ground_data = [ground_arctic_day, ground_arctic_night, ground_antarctic_day, ground_antarctic_night]

## Compare group averages on the ground

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

colors = ["blue", "orange", "green", "red", "purple", "pink"]

for i, df in enumerate(ground_data):
    grouped = df.groupby("group")[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]].mean()

    for j, column in enumerate(grouped.columns):
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second", color=colors[j])

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} on the ground")
    axes[i].set_ylabel("Particles Per Second (Log)")

fig.suptitle("Group averages for all sensors for in_shadow on the ground", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", fontsize="large")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 8), constrained_layout=True)

for i, df in enumerate(ground_data):
    grouped = df.groupby("group")[["xray0_ps", "proton0_ps", "electron0_ps"]].mean()

    for column in grouped.columns:
        axes[i].plot(grouped.index, np.log(grouped[column]), label=f"{column[:-3]} per second")

    axes[i].set_title(f"Radiation in the {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} on the ground", fontsize="x-large")
    axes[i].set_ylabel("Particles Per Second (Log)")


fig.suptitle("Group averages for main sensors for in_shadow on the ground", fontsize="xx-large")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", bbox_to_anchor=(1.002, 1.02), fontsize="medium")
plt.show()

## Compare sensors on the ground with a box and whisker plot

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

axes = axes.flatten()

for i, df in enumerate(ground_data):

    subset = df[["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

    bp = axes[i].boxplot(subset,
                         tick_labels=subset.keys(),
                         patch_artist=True,
                         boxprops=dict(facecolor="steelblue", alpha=0.6),
                         medianprops=dict(color="black", linewidth=2),
                         flierprops=dict(marker="o", markersize=2, alpha=0.3)
    )

    for median_line in bp['medians']:
        median_val = median_line.get_ydata()[0]
        left_edge_x = median_line.get_xdata()[0]

        axes[i].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)
    
    axes[i].set_yscale("log")
    axes[i].set_ylim(10**-3, 10**5)
    axes[i].set_ylabel("Particles Per Second (Log)")
    axes[i].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(df)})")

# fig.delaxes(axes[3])
plt.suptitle("All sensors per second by pole and daylight on the ground", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of box and whisker plot on the ground

It appears that a lot of the low values for `xray0_ps` where `in_shadow == 0` fall under `ground_shadow == 1`. This could be a coincidence, but it looks like sunlight hitting the surface may have a potential effect on X-ray readings. This could be due to X-rays reflecting back up from the surface, or, as mentioned, it could be complete chance. More analysis would be required to know for sure. Barring that, most things stayed fairly similar between box plots.

## Compare ground daylight to satellite daylight

In [ ]:
all_data = [(arctic_day, ground_arctic_day), 
            (arctic_night, ground_arctic_night), 
            (antarctic_day, ground_antarctic_day), 
            (ground_antarctic_night, ground_antarctic_night)]

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(12, 12))

axes = axes.flatten()

for i, dfs in enumerate(all_data):
    for j in range(2):
        subset = dfs[j][["xray0_ps", "xray1_ps", "xray2_ps", "xray3_ps", "proton0_ps", "electron0_ps"]]

        bp = axes[2*i+j].boxplot(subset,
                            tick_labels=subset.keys(),
                            patch_artist=True,
                            boxprops=dict(facecolor="steelblue", alpha=0.6),
                            medianprops=dict(color="black", linewidth=2),
                            flierprops=dict(marker="o", markersize=2, alpha=0.3)
        )

        for median_line in bp['medians']:
            median_val = median_line.get_ydata()[0]
            left_edge_x = median_line.get_xdata()[0]

            axes[2*i+j].hlines(y=median_val, xmin=axes[i].get_xlim()[0], xmax=left_edge_x, color='gray', linestyle='--', linewidth=1)

        
        axes[2*i+j].set_yscale("log")
        axes[2*i+j].set_ylim(10**-3, 10**5)
        axes[2*i+j].set_title(f"Radiation in {"Arctic" if i < 2 else "Antarctic"} Circle when {"satellite" if (j % 2 == 0) else "ground"} in {"shadow" if (i % 2 != 0) else "sunlight"} (Data points: {len(dfs[j])})", fontsize="medium")

axes[6].set_visible(False)
plt.suptitle("All sensors per second by pole and daylight on the ground", fontsize="xx-large")
plt.tight_layout()
plt.show()

### Analysis of ground daylight vs satellite daylight

This is an interesting graph, but I'm not sure how helpful it really is. Sample size issues make it fairly hard to compare graphs to each other, we don't have enough points to analyze the Antarctic Circle with the satellite in shadow with any integrity, and there is significant overlap in the `in_shadow` and `ground_shadow` columns. I created it hoping to see some sort of dramatic difference, but there isn't one. 
  
`xray0_ps` shows an order of magnitude difference between satellite sunlight and ground sunlight (and the same for the lack thereof), but these, again, could come from diffrences in sample sizes and overlaps between the data. Ground sunlight is effectively just a smaller version of satellite sunlight here. Same with ground shadow and satellite shadow. With those issues and `xray0_ps` being the only column showing any differences, I'm concerned about drawing any conclusions here. Maybe the experts will have something else to say about it, though.